# Train U-Net on Euclid-like dataset

This notebook trains the `UNet2Dto3D` model on the complex, noisy, multi-order
euclid-like dataset generated by `ComplexSimulator`.

## Key differences from the miniature version
- **Noise**: Poisson sky background + read noise + dark current (NISP model).
- **Multi-order**: 0th-order peanut, 1st and 2nd diffraction orders.
- **SEDs**: Blackbody or configurable realistic templates.
- **5-channel input**: spectrograms (4ch) + direct undispersed image (1ch).

## Architecture
The `UNet2Dto3D` takes a 2-D multi-channel input `(B, C_in, H_spec, W_spec)`
and upsamples/expands it into a 3-D spectral cube `(B, n_lambda, ny, nx)` via
encoder–decoder skip connections.  The transition from 2-D to 3-D happens in
the bottleneck: the feature map is reshaped and passed through 3-D deconvolution
blocks that simultaneously restore spatial resolution and expand along λ.

In [ ]:
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt

from spectangle.data_loaders.dataset import SpectangleDataModule
from spectangle.models.unet import UNet2Dto3D
from spectangle.models.losses import CombinedLoss
from spectangle.utils.metrics import cube_metrics
from spectangle.utils.training import get_device

# ── Config ─────────────────────────────────────────────────────────────────
IN_CHANNELS = 5        # 4 spectrograms + 1 direct image
N_LAMBDA    = 128
BASE_CH     = 32       # base channel width (increase to 64 for full run)
LR          = 3e-4
N_EPOCHS    = 8
BATCH_SIZE  = 4
DEVICE      = get_device()
print(f'Device: {DEVICE}')

## 1. Load data

In [ ]:
dm = SpectangleDataModule(
from spectangle.paths import RAW_DIR
H5_PATH       = RAW_DIR / 'complex_euclid_200s.h5'
    batch_size=BATCH_SIZE,
    n_channels=IN_CHANNELS,
    normalise='per_sample',
    num_workers=0,
    seed=42,
)
print(dm)

scene_shape       = dm.scene_shape
spectrogram_shape = dm.spectrogram_shape

# Inspect one batch
x_batch, y_batch = next(iter(dm.train_dataloader()))
print(f'Input  batch: {x_batch.shape}  dtype={x_batch.dtype}')
print(f'Target batch: {y_batch.shape}  dtype={y_batch.dtype}')
print(f'Input  range: [{x_batch.min():.3f}, {x_batch.max():.3f}]')
print(f'Target range: [{y_batch.min():.3f}, {y_batch.max():.3f}]')

## 2. Build the U-Net model

In [ ]:
model = UNet2Dto3D(
    in_channels=IN_CHANNELS,
    n_lambda=N_LAMBDA,
    base_features=BASE_CH,
    scene_shape=scene_shape,
).to(DEVICE)

print(f'Parameters: {model.parameter_count():,}')

# Sanity check
x_dummy = torch.zeros(2, IN_CHANNELS, *spectrogram_shape, device=DEVICE)
y_dummy = model(x_dummy)
print(f'Output shape: {y_dummy.shape}  (expected: [2, {N_LAMBDA}, {scene_shape[0]}, {scene_shape[1]}])')

## 3. Train

In [ ]:
loss_fn   = CombinedLoss(weights={'mse': 1.0, 'ssim': 0.5, 'spectral': 0.05})
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR, epochs=N_EPOCHS,
    steps_per_epoch=len(dm.train_dataloader()),
)

def run_epoch(loader, train):
    model.train() if train else model.eval()
    total = 0.0; n = 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            pred = model(x)
            loss, _ = loss_fn(pred, y)
            if train:
                optimizer.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step(); scheduler.step()
            total += loss.item() * x.shape[0]; n += x.shape[0]
    return total / max(n, 1)

tl_hist, vl_hist = [], []
for ep in range(1, N_EPOCHS + 1):
    tl = run_epoch(dm.train_dataloader(), True)
    vl = run_epoch(dm.val_dataloader(),   False)
    tl_hist.append(tl); vl_hist.append(vl)
    print(f'Epoch {ep:3d}/{N_EPOCHS} | train={tl:.5f} | val={vl:.5f}')

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(tl_hist, label='train'); ax.plot(vl_hist, label='val')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.legend(); ax.grid(True)
ax.set_title('U-Net training curves (Euclid-like complex)')
plt.tight_layout(); plt.show()

## 4. Visual reconstruction comparison

In [ ]:
model.eval()
x_s, y_s = dm._val_ds[0]
with torch.no_grad():
    pred = model(x_s.unsqueeze(0).to(DEVICE)).squeeze(0).cpu().numpy()
gt = y_s.numpy()

metrics = cube_metrics(pred, gt)
print('Reconstruction metrics (val[0]):')
for k, v in metrics.items():
    print(f'  {k}: {v:.4f}')

# Broadband
fig, axs = plt.subplots(1, 3, figsize=(13, 4))
axs[0].imshow(gt.sum(0),   cmap='gray',  origin='lower'); axs[0].set_title('GT broadband')
axs[1].imshow(pred.sum(0), cmap='gray',  origin='lower'); axs[1].set_title('U-Net prediction')
axs[2].imshow(np.abs(pred.sum(0)-gt.sum(0)), cmap='magma', origin='lower'); axs[2].set_title('|Residual|')
for a in axs: a.axis('off')
plt.tight_layout(); plt.show()

# 6 wavelength slices
lam_idx = np.linspace(0, N_LAMBDA-1, 6, dtype=int)
wav = np.linspace(9250, 18500, N_LAMBDA)
fig, axs = plt.subplots(2, 6, figsize=(15, 5))
for col, li in enumerate(lam_idx):
    axs[0, col].imshow(gt[li],   cmap='gray', origin='lower'); axs[0, col].set_title(f'λ={wav[li]:.0f}Å\nGT')
    axs[1, col].imshow(pred[li], cmap='gray', origin='lower'); axs[1, col].set_title('Pred')
    axs[0, col].axis('off'); axs[1, col].axis('off')
plt.suptitle('U-Net: wavelength slices (top=GT, bottom=pred)')
plt.tight_layout(); plt.show()

## 5. Spectral fidelity at source positions
Extract recovered spectra at the positions of the brightest sources and
compare them to the ground-truth Blackbody SEDs.

In [ ]:
import h5py
# Load source positions from the HDF5
from spectangle.paths import RAW_DIR
H5_PATH       = RAW_DIR / 'complex_euclid_200s.h5'
    keys = sorted(f['samples'].keys())
    # Find the val sample index corresponding to dm._val_ds[0]
    val_key = dm._val_ds._keys[0]
    grp = f['samples'][val_key]
    src_xs = grp['source_xs'][:]
    src_ys = grp['source_ys'][:]

fig, ax = plt.subplots(figsize=(8, 3))
for i, (sx, sy) in enumerate(zip(src_xs[:3], src_ys[:3])):
    ix, iy = int(round(sx)), int(round(sy))
    iy = min(max(iy, 0), gt.shape[1]-1)
    ix = min(max(ix, 0), gt.shape[2]-1)
    ax.plot(wav, gt[:, iy, ix],   label=f'GT src{i}',   lw=1.5)
    ax.plot(wav, pred[:, iy, ix], label=f'Pred src{i}', lw=1.2, ls='--')

ax.set_xlabel('Wavelength (Å)'); ax.set_ylabel('Flux (a.u.)')
ax.legend(fontsize=7); ax.grid(True)
ax.set_title('Recovered spectra at source positions (top 3 sources)')
plt.tight_layout(); plt.show()

## Next steps
- Run full training via: `python scripts/train.py --config configs/models/unet.yaml`
- Compare with the PINN: `notebooks/euclid_like_version/training_testing/01_train_pinn_on_euclid_like.ipynb`
- Increase `BASE_CH=64` and `N_EPOCHS=200` for publication-quality results.